<img src="https://upload.wikimedia.org/wikipedia/commons/8/89/TheNorthFace_logo.svg" alt="THE NORTH FACE LOGO" width="50%" />

# The North Face e-commerce : boosting online sales !

## Company's Description 📇

The North Face is an American outdoor recreation product company, founded in 1968 to supply climbers. The North Face produces clothing, footwear, and outdoor equipment. By the late 1990s customers had expanded beyond outdoor enthusiasts and in the 2000s it became a fashion style symbol.


## Project 🚧

The marketing department would like to take advantage of machine learning solutions to boost online sales on the website : https://www.thenorthface.fr/

They have identified two major solutions that could have a huge effect on the conversion rates :
- Deploying a **recommender system** that will allow to suggest additionnal products to users, that are similar to the items they are already interested in. The recommendations could be materialized by a "you might also be interested by these products..." section that would appear on each product page of the website.
- Improving the structure of the products catalog thanks to **topic extraction**. The idea is to use unsupervised methods to challenge the existing categories : is it possible to find new categories of product that would be more suitable for the navigation on the website ?

## Goals 🎯

The project can be cut into three steps :

1. Identify groups of products that have similar descriptions.

2. Use the groups of similar products to build a simple recommender system algorithm.

3. Use topic modeling algorithms to automatically assess the latent topics present in the item descriptions.

## Scope of this project 🖼️

In this project, you'll work with a corpus of item descriptions from The North Face's product catalog. Data can be found here: 

👉👉<a href="https://www.kaggle.com/cclark/product-item-data?select=sample-data.csv" target="_blank"> The North Face product catalog</a> 👈👈


## Helpers 🦮

To help you achieve this project, here are a few tips that should help you: 

### Preprocessing of textual data

- First, you'll have to make some preprocessings to clean the corpus (in particular, don't forget to handle stop words and to lemmatize the documents). For this step, you'll need `pandas` and `spacy`.

- Then, you can to encode the texts with **TF-IDF transformation** (this can be done with scikit learn's `TFIDFVectorizer`)

### Part 1 : Groups of products with similar descriptions

For this part, you'll have to train a **clustering model** that will create groups of products for which the descriptions are "close" to each other. 
- You can use `DBSCAN` to make some clustering on the TF-IDF matrix. When dealing with texts, the distance metric to be used is "cosine" instead of "euclidean". You have to find some values for `eps` and `min_samples` that allow to get 10-20 clusters and not too many outliers.
- You can display a `wordcloud` for each cluster to analyze the results and see if the groups contain different words.

### Part 2 - Recommender system

Then, you can use the cluster ids from part 1 to build a **recommender system**. The aim is to be able to suggest to a user some products that are similar to the ones he is interested in. To do this, we will consider that products belonging to the same cluster are similar.

- Create a function named `find_similar_items` that takes an argument `item_id` representing the id of a product, and that returns a list of 5 item ids that belong to the same cluster as the product passed in argument
- Use python's `input()` function to allow the user to choose a product and perform some suggestions of similar items

### Part 3 : Topic modeling

*This part is independant from the two others.*

The aim is to use an LSA model to automatically extract latent topics in the products' descriptions.

- Use TruncatedSVD to make some topic extraction from the TF-IDF matrix. You can try values for `n_components` that allow to get 10-20 topics. Save the encoded matrix into a variable named `topic_encoded_df`.
- Contrary to clustering, LSA allows to map each document to a mixing of several topics. For this reason, it's a bit more difficult to interpret the topics as categories : one document can actually be related to several topics at a time. To make things easier, consider extracting the main topic of each document.
- As in part 1, you can display `wordclouds` to analyze the results.

## Deliverable 📬

To complete this project, your team should: 

- Train at least one **clustering model** on the corpus and display wordclouds describing the clusters
- Develop a simple python code that allows a user to type the id of a product he's interested in, and then get a list of similar items
- Train at least one **TruncatedSVD model** on the corpus and display wordclouds describing the latent topics


# Exploration des données

In [1]:
import pandas as pd
import spacy

import plotly.express as px
import matplotlib.pyplot as plt
from wordcloud import WordCloud

In [2]:
df = pd.read_csv("sample-data.csv")

print(df.shape)

(500, 2)


500 lignes, et 2 colonnes.

In [3]:
print(df.isna().sum())

id             0
description    0
dtype: int64


Pas de valeur manquante, ni dans la colonne ID, ni dans la colonne description

In [4]:
print("Descriptions en double :", df["description"].duplicated().sum())
df.head(20)

Descriptions en double : 0


,id,description
0,1,Active classic boxers - There's a reason why o...
1,2,Active sport boxer briefs - Skinning up Glory ...
2,3,Active sport briefs - These superbreathable no...
3,4,"Alpine guide pants - Skin in, climb ice, switc..."
4,5,"Alpine wind jkt - On high ridges, steep ice an..."
5,6,Ascensionist jkt - Our most technical soft she...
6,7,"Atom - A multitasker's cloud nine, the Atom pl..."
7,8,Print banded betina btm - Our fullest coverage...
8,9,Baby micro d-luxe cardigan - Micro D-Luxe is a...
9,10,Baby sun bucket hat - This hat goes on when th...


Pas de description en double

On peut maintenant s'interesser a la longueur typique d'une description

In [5]:
px.histogram(df["description"].str.len(), nbins=40,
             labels={"value": "Longueur de la description (caractères)"})

In [6]:
df["description"].str.len().describe()

count     500.000000
mean     1122.180000
std       424.608191
min       416.000000
25%       798.500000
50%      1046.000000
75%      1393.500000
max      3540.000000
Name: description, dtype: float64

Les descriptions vont de 400 à 3500 caracteres environ. La description a 3500 caractere fait office d'outlier et est inahbituellement longue. La seconde description la plus loingue est a 2700.
La mediane est a 1000 caracteres environ.

On peut regarder une description entiere pour voir à quoi elle ressemble.

In [8]:
print(df["description"].iloc[0])

Active classic boxers - There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2". Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>"Silky Capilene 1 fabric is ultralight, breathable and quick-to-dry"</li> <li>"Exposed, brushed elastic waistband for comfort"</li> <li>5-panel construction with traditional boxer back</li> <li>"Inseam (size M) is 4 1/2"""</li></ul><br><br><b>Fabric: </b>3.7-oz 100% all-recycled polyester with Gladiodor natural odor control for the garment. Recyclable through the Common Threads Recycling Program<br><br><b>Weight: </b>99 g (3.5 oz)<br><br>

On observe que les descriptions sont livrées en format HTML. On peut y voir notamment les balises de paragraphe, mise en gras ou autre. C'est particulièrement visible à la fin de la description. Il va donc d'abord être nécessaire de nettoyer le corpus des balises HTML avant de passer à son traitement. 

# Preparation du corpus

Nous allons donc procéder au nettoyage du corpus en enlevant, d'une part, toute balise HTML (c'est-à-dire ce qui est compris entre deux chevrons), mais aussi en retirant tout ce qui n'est ni une lettre ni un espace. On ne conserve au final que les mots, et on les met en minuscules. Tous les caractères enlevés sont remplacés par un espace.   
On utilise pour cela du regex.

In [10]:
df["clean_description"] = (df["description"]
                           .str.replace(r"<[^>]+>", " ", regex=True)
                           .str.replace(r"[^A-Za-z ]+", " ", regex=True)
                           .str.lower())

print(df["clean_description"].iloc[0])

active classic boxers   there s a reason why our boxers are a cult favorite   they keep their cool  especially in sticky situations  the quick drying  lightweight underwear takes up minimal space in a travel pack  an exposed  brushed waistband offers next to skin softness  five panel construction with a traditional boxer back for a classic fit  and a functional fly  made of  oz   recycled polyester with moisture wicking performance  inseam  size m  is     recyclable through the common threads recycling program    details      silky capilene   fabric is ultralight  breathable and quick to dry     exposed  brushed elastic waistband for comfort     panel construction with traditional boxer back    inseam  size m  is         fabric    oz   all recycled polyester with gladiodor natural odor control for the garment  recyclable through the common threads recycling program   weight     g   oz   made in mexico 


Le texte est maintenant nettoyé des balises ainsi que de la ponctuation. 
Pour pouvoir le vectoriser, deux étapes sont encore nécessaires : 
- la lemmatisation
- le retrait des mots vides (stop words). 

- La lemmatisation va permettre de ramener chaque mot a la forme sous laquelle on le trouverait dans un dictionnaire: mots au singulier, verbes à l'infinitif. 
- Les mots vides sont les mots tels que "and" et "the", etc... Ils reviennent fréquemment, sont porteurs de peu d'informations et ne permettent aucune differenciation entre les descriptions.  

In [13]:
nlp = spacy.load("en_core_web_sm")

def lemmatiser(texte):
    doc = nlp(texte)   # La lemmatisation se fait ici
    lemmes = [token.lemma_ for token in doc if not token.is_stop and not token.is_space]   # On filtre les Lemme en retirant ceux qui sont des stop words ou des espaces
    return " ".join(lemmes)

df["tokens"] = df["clean_description"].apply(lemmatiser)

print(df["tokens"].iloc[0])

active classic boxer s reason boxer cult favorite cool especially sticky situation quick dry lightweight underwear take minimal space travel pack expose brush waistband offer skin softness panel construction traditional boxer classic fit functional fly oz recycle polyester moisture wicke performance inseam size m recyclable common thread recycle program detail silky capilene fabric ultralight breathable quick dry expose brush elastic waistband comfort panel construction traditional boxer inseam size m fabric oz recycle polyester gladiodor natural odor control garment recyclable common thread recycle program weight g oz mexico
